In [0]:
from pyspark.sql import functions as F, Window

df_fact = spark.table("workspace.gold.fact_movies_performance")

resultado_1 = df_fact.agg(
    F.format_number(F.sum("receita_brl"), 2).alias("receita_total_brl")
)

display(resultado_1)

receita_total_brl
"834,732,290,730.33"


In [0]:
df_movies = spark.table("workspace.gold.dim_movies")

resultado_2 = (df_fact
    .join(df_movies, on="sk_movie_id")
    .select("titulo", "popularidade")
    .orderBy(F.desc("popularidade"))
    .limit(5))

display(resultado_2)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
retribution,1547.22


In [0]:
df_bridge_genre = spark.table("workspace.gold.bridge_movie_genre")
df_genres = spark.table("workspace.gold.dim_genres")

resultado_3 = (df_bridge_genre
    .join(df_genres, on="sk_genre_id")
    .groupBy("nome_genero")
    .agg(F.count("*").alias("qtd_filmes"))
    .orderBy(F.desc("qtd_filmes")))

display(resultado_3)

nome_genero,qtd_filmes
Drama,32250
Documentary,18981
Comedy,18585
Thriller,10264
Horror,9715
Romance,7628
Action,6035
Crime,4733
Animation,4457
TV Movie,4070


In [0]:
df_metricas_gold = spark.table("workspace.silver.tb_metricas_engajamento")

df_metricas_gold.filter(
    (F.col("popularidade") >= 1900) & (F.col("popularidade") <= 2030) &
    (F.col("popularidade") == F.floor(F.col("popularidade")))
).count()

0

In [0]:
window_rank = Window.orderBy(F.desc("receita_usd"))

resultado_4 = (df_fact
    .join(df_movies, on="sk_movie_id")
    .filter(F.col("receita_usd").isNotNull())
    .select(
        "titulo",
        F.format_number("receita_usd", 2).alias("receita_usd"),
        F.format_number("receita_brl", 2).alias("receita_brl"),
        F.rank().over(window_rank).alias("posicao_ranking")
    )
    .orderBy("posicao_ranking")
    .limit(10))

display(resultado_4)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


titulo,receita_usd,receita_brl,posicao_ranking
Avengers: Endgame,"2,800,000,000.00","14,439,320,000.00",1
Avatar: The Way of Water,"2,320,250,281.00","11,965,298,674.09",2
AVENGERS: INFINITY WAR,"2,052,415,039.00","10,584,099,114.62",3
spider-man: no way home,"1,921,847,111.00","9,910,773,366.72",4
The Lion King,"1,663,075,401.00","8,576,313,535.42",5
Top Gun: Maverick,"1,488,732,821.00","7,677,246,284.61",6
Barbie,"1,428,545,028.00","7,366,863,854.89",7
The Super Mario Bros. Movie,"1,355,725,263.00","6,991,339,608.76",8
Black Panther,"1,349,926,083.00","6,961,433,817.42",9
Star Wars: The Last Jedi,"1,332,698,830.00","6,872,594,596.43",10


In [0]:
from datetime import date

hoje = F.current_date()

data_limite = (df_movies
    .filter((F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= hoje))
    .agg(F.max("data_lancamento"))
    .collect()[0][0])

print(f"Data limite (mais recente lançamento realizado): {data_limite}")

data_limite_2anos = F.date_sub(F.lit(data_limite), 365 * 2)

df_bridge_person_gold = spark.table("workspace.gold.bridge_movie_person")
df_people_gold = spark.table("workspace.gold.dim_people")

resultado_5 = (df_bridge_person_gold
    .join(df_people_gold, on="sk_person_id")
    .filter(F.col("tipo_pessoa") == "Ator")
    .join(df_movies.select("sk_movie_id", "data_lancamento", "status_filme"), on="sk_movie_id")
    .filter(
        (F.col("status_filme") == "Lançado") &
        (F.col("data_lancamento") >= data_limite_2anos) &
        (F.col("data_lancamento") <= F.lit(data_limite))
    )
    .groupBy("nome_pessoa")
    .agg(F.count("*").alias("qtd_participacoes"))
    .orderBy(F.desc("qtd_participacoes"))
    .limit(1))

display(resultado_5)

Data limite (mais recente lançamento realizado): 2026-02-19


nome_pessoa,qtd_participacoes
Adlih Torres,12


In [0]:
data_limite_5anos = F.date_sub(F.lit(data_limite), 365 * 5)

df_bridge_company_gold = spark.table("workspace.gold.bridge_movie_company")
df_companies_gold = spark.table("workspace.gold.dim_companies")

resultado_6 = (df_bridge_company_gold
    .join(df_companies_gold, on="sk_company_id")
    .join(df_movies.select("sk_movie_id", "data_lancamento", "status_filme"), on="sk_movie_id")
    .join(df_fact.select("sk_movie_id", "lucro_usd"), on="sk_movie_id")
    .filter(
        (F.col("status_filme") == "Lançado") &
        (F.col("data_lancamento") >= data_limite_5anos) &
        (F.col("data_lancamento") <= F.lit(data_limite)) &
        (F.col("lucro_usd").isNotNull())
    )
    .groupBy("nome_produtora")
    .agg(F.round(F.sum("lucro_usd"), 2).alias("lucro_total_usd"))
    .orderBy(F.desc("lucro_total_usd"))
    .limit(1))

display(resultado_6)

nome_produtora,lucro_total_usd
Universal Pictures,5.772329679E9
